In [ ]:
import os
if os.path.exists('/content/dataset.zip'):
    !unzip /content/dataset.zip
else:
    print('dataset.zip not found. Please ensure the file is uploaded.')

Streaming output truncated to the last 5000 lines.
  inflating: dataset/train/1/30674.png  
  inflating: dataset/train/1/12671.png  
  inflating: dataset/train/1/02626.png  
  inflating: dataset/train/1/09599.png  
  inflating: dataset/train/1/26276.png  
  inflating: dataset/train/1/12247.png  
  inflating: dataset/train/1/31953.png  
  inflating: dataset/train/1/45642.png  
  inflating: dataset/train/1/55179.png  
  inflating: dataset/train/1/10156.png  
  inflating: dataset/train/1/09595.png  
  inflating: dataset/train/1/56646.png  
  inflating: dataset/train/1/64753.png  
  inflating: dataset/train/1/04530.png  
  inflating: dataset/train/1/23225.png  
  inflating: dataset/train/1/10486.png  
  inflating: dataset/train/1/48562.png  
  inflating: dataset/train/1/18837.png  
  inflating: dataset/train/1/63022.png  
  inflating: dataset/train/1/43987.png  
  inflating: dataset/train/1/22520.png  
  inflating: dataset/train/1/69203.png  
  inflating: dataset/train/1/05957.png  
  infl

In [ ]:
# --- BLOCK: MODEL DEFINITION (roll_no.py) ---
import torch
import torch.nn as nn
from torchvision import models

class MyAgeClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = models.resnet18(weights=None)

        num_ftrs = self.backbone.fc.in_features

        self.backbone.fc = nn.Sequential(
            nn.BatchNorm1d(num_ftrs),
            nn.Dropout(0.4),
            nn.Linear(num_ftrs, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        if self.training:
            return self.backbone(x)
        else:
            out_original = self.backbone(x)
            out_flipped = self.backbone(torch.flip(x, dims=[3]))
            return (out_original + out_flipped) / 2.0


In [ ]:
# --- BLOCK: MODEL DEFINITION (roll_no.py) ---
import torch
import torch.nn as nn
from torchvision import models

class MyAgeClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = models.resnet18(weights=None)

        num_ftrs = self.backbone.fc.in_features

        self.backbone.fc = nn.Sequential(
            nn.BatchNorm1d(num_ftrs),
            nn.Dropout(0.4),
            nn.Linear(num_ftrs, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        if self.training:
            return self.backbone(x)
        else:
            out_original = self.backbone(x)
            out_flipped = self.backbone(torch.flip(x, dims=[3]))
            return (out_original + out_flipped) / 2.0

# --- BLOCK: IMPORTS & SETUP ---
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
import torch.optim.swa_utils as swa_utils

# --- BLOCK: CUSTOM VALID DATASET ---
class ValidDataset(Dataset):
    def __init__(self, valid_dir, csv_file, transform=None):
        self.valid_dir = valid_dir
        self.transform = transform
        self.samples = []

        with open(csv_file, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(',')
                if len(parts) >= 2 and parts[1].strip().isdigit():
                    self.samples.append((parts[0].strip(), int(parts[1].strip())))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, label = self.samples[idx]
        path = os.path.join(self.valid_dir, fname)
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# --- BLOCK: MIXUP UTILITY ---
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = torch.distributions.Beta(alpha, alpha).sample().item()
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# --- BLOCK: TRAINING SCRIPT ---
def train_model(data_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_dir = os.path.join(data_dir, 'train')
    valid_dir = os.path.join(data_dir, 'valid')
    valid_csv = os.path.join(data_dir, 'valid_labels.csv')

    train_transforms = T.Compose([
        T.RandomResizedCrop(224, scale=(0.7, 1.0)),
        T.RandomHorizontalFlip(),
        T.RandAugment(num_ops=2, magnitude=5),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        T.RandomErasing(p=0.2)
    ])

    train_dataset = ImageFolder(root=train_dir, transform=train_transforms)
    valid_dataset = ValidDataset(valid_dir=valid_dir, csv_file=valid_csv, transform=train_transforms)

    combined_dataset = ConcatDataset([train_dataset, valid_dataset])
    print(f"Total training images (Train + Valid): {len(combined_dataset)}")

    train_loader = DataLoader(combined_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)

    model = MyAgeClassifier().to(device)

    ema_model = swa_utils.AveragedModel(
        model,
        multi_avg_fn=swa_utils.get_ema_multi_avg_fn(0.9997)
    )

    criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    epochs = 80
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=3e-3,
        steps_per_epoch=len(train_loader),
        epochs=epochs,
        pct_start=0.2
    )

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            inputs, targets_a, targets_b, lam = mixup_data(inputs, targets, alpha=0.2)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()
            ema_model.update_parameters(model)

            epoch_loss += loss.item()

        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f}")

    torch.save(ema_model.module, 'b23cm1003.pth')
    print("Training complete. Model saved as 'b23cm1003.pth'")

# --- BLOCK: EXECUTION ---
if __name__ == "__main__":
    DATASET_PATH = '/content/dataset'
    train_model(DATASET_PATH)

Using device: cuda
Total training images (Train + Valid): 18466


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 01/80 | Loss: 0.5809
Epoch 02/80 | Loss: 0.4676
Epoch 03/80 | Loss: 0.4501
Epoch 04/80 | Loss: 0.4400
Epoch 05/80 | Loss: 0.4257
Epoch 06/80 | Loss: 0.4194
Epoch 07/80 | Loss: 0.3859
Epoch 08/80 | Loss: 0.4123
Epoch 09/80 | Loss: 0.3845
Epoch 10/80 | Loss: 0.3899
Epoch 11/80 | Loss: 0.3772
Epoch 12/80 | Loss: 0.3748
Epoch 13/80 | Loss: 0.3850
Epoch 14/80 | Loss: 0.3839
Epoch 15/80 | Loss: 0.3657
Epoch 16/80 | Loss: 0.3736
Epoch 17/80 | Loss: 0.3618
Epoch 18/80 | Loss: 0.3682
Epoch 19/80 | Loss: 0.3576
Epoch 20/80 | Loss: 0.3427
Epoch 21/80 | Loss: 0.3469
Epoch 22/80 | Loss: 0.3434
Epoch 23/80 | Loss: 0.3437
Epoch 24/80 | Loss: 0.3374
Epoch 25/80 | Loss: 0.3484
Epoch 26/80 | Loss: 0.3346
Epoch 27/80 | Loss: 0.3496
Epoch 28/80 | Loss: 0.3355
Epoch 29/80 | Loss: 0.3255
Epoch 30/80 | Loss: 0.3255
Epoch 31/80 | Loss: 0.3220
Epoch 32/80 | Loss: 0.3271
Epoch 33/80 | Loss: 0.3186
Epoch 34/80 | Loss: 0.3240
Epoch 35/80 | Loss: 0.3233
Epoch 36/80 | Loss: 0.3143
Epoch 37/80 | Loss: 0.3187
E

In [ ]:
from google.colab import files
files.download('b23cm1003.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>